# Instalment Payments – Feature Engineering

Builds customer-level features from the `installments_payments` dataset for the Home Credit default prediction pipeline.

**Pipeline overview**
1. Load raw data & filter to training customers
2. Derive `MONTHS_BALANCE` and sort
3. Impute missing values
4. Derive instalment-level features (late days, payment ratio, version change)
5. Monthly roll-up: sums & maxes
6. Loan-level aggregations (all-time & last 1Y)
7. Enrich with PrevLoans metadata
8. Time-window aggregations (1Y, 3M) + last snapshot
9. Merge all features → save

## 1. Imports

In [1]:
import os
import pandas as pd
import numpy as np
import gc

## 2. Configuration & Paths

In [2]:
DATA_ROOT = os.environ.get("HOMECREDIT_DATA_ROOT", r"a:\Dominic\My Projects\Home Credit\data")

PATH_PAYMENTS    = os.path.join(DATA_ROOT, "Input", "installments_payments.csv")
PATH_APP_TRAIN   = os.path.join(DATA_ROOT, "Input", "application_train.csv")
PATH_PREV_LOANS  = os.path.join(DATA_ROOT, "Model-Training", "Staging", "prev_loan_data_processed.parquet")
PATH_OUTPUT      = os.path.join(DATA_ROOT, "Model-Training", "Staging", "payment_installments_processed.parquet")

## 3. Load Data

In [3]:
base = pd.read_csv(PATH_APP_TRAIN, usecols=['SK_ID_CURR', 'TARGET'])
base = base.astype({'SK_ID_CURR': 'string'})

Payments = pd.read_csv(PATH_PAYMENTS)
Payments = Payments.astype({'SK_ID_PREV': 'string', 'SK_ID_CURR': 'string'})

# Restrict to training customers; keep TARGET for final merge
Payments = base.merge(Payments, on='SK_ID_CURR', how='inner').drop(columns=['TARGET'])

# Save unique customer list before any deletions — used as merge spine later
unique_customers = Payments[['SK_ID_CURR']].drop_duplicates()

print(f"Payments shape: {Payments.shape}")

Payments shape: (11591592, 8)


## 4. Derive `MONTHS_BALANCE` & Sort

`DAYS_INSTALMENT` is a negative integer (days relative to the application date).
We bucket it into calendar months using `np.floor` division.

> **Fix:** the original used `.astype('int')` which truncates toward zero
> (e.g. `-12.9 → -12`). `np.floor` correctly rounds toward negative infinity
> (`-12.9 → -13`), giving consistent same-month bucketing for negative values.

The sort includes both `MONTHS_BALANCE` **and** `DAYS_ENTRY_PAYMENT` so that
within a month, records are ordered by actual payment date — important for
`.first()` / `.last()` operations downstream.

In [4]:
# FIX: use floor division instead of int truncation for negative day values
Payments['MONTHS_BALANCE'] = np.floor(Payments['DAYS_INSTALMENT'] / 30).astype(int)

Payments.sort_values(
    by=['SK_ID_CURR', 'SK_ID_PREV', 'MONTHS_BALANCE', 'DAYS_ENTRY_PAYMENT'],
    inplace=True,
    ignore_index=True,
)

print(f"MONTHS_BALANCE range: [{Payments['MONTHS_BALANCE'].min()}, {Payments['MONTHS_BALANCE'].max()}]")

MONTHS_BALANCE range: [-98, -1]


## 5. Impute Missing Values

Forward-fill then back-fill (limit = 6 months) within each loan (`SK_ID_PREV`)
for payment date and payment amount.

In [5]:
for col in ['DAYS_ENTRY_PAYMENT', 'AMT_PAYMENT']:
    Payments[col] = (
        Payments.groupby('SK_ID_PREV')[col]
        .transform(lambda x: x.ffill(limit=6).bfill(limit=6))
    )

print("Remaining nulls after imputation:")
print(Payments[['DAYS_ENTRY_PAYMENT', 'AMT_PAYMENT']].isna().sum().to_string())

Remaining nulls after imputation:
DAYS_ENTRY_PAYMENT    508
AMT_PAYMENT           508


## 6. Derive Previous Instalment-Level Features

- `LATE_PAYMENT_DAYS`: positive = payment made after due date; negative = early.
- `PAYMENT_AMT_TO_INSTALLMENT_RATIO`: how much of the scheduled amount was actually paid.
- `LATE_PAYMENT_FLAG`: 1 if payment was late, else 0.
- `INSTALMENT_VERSION_CHANGE`: flags months where the repayment schedule was renegotiated
  (i.e. `NUM_INSTALMENT_VERSION` changed).

In [6]:
Payments['LATE_PAYMENT_DAYS'] = (
    Payments['DAYS_ENTRY_PAYMENT'] - Payments['DAYS_INSTALMENT']
)
Payments['PAYMENT_AMT_TO_INSTALLMENT_RATIO'] = np.where(
    Payments['AMT_INSTALMENT'] > 0,
    Payments['AMT_PAYMENT'] / Payments['AMT_INSTALMENT'],
    np.nan,
)
Payments['LATE_PAYMENT_FLAG'] = (
    Payments['LATE_PAYMENT_DAYS'] > 0
).astype(int)

# Flags rows where the instalment schedule version changed within a loan
Payments['INSTALMENT_VERSION_CHANGE'] = (
    Payments.groupby('SK_ID_PREV')['NUM_INSTALMENT_VERSION']
    .diff()
    .fillna(0)
    .astype(int)
)

print("Late payment rate:",
      f"{Payments['LATE_PAYMENT_FLAG'].mean():.2%}")

Late payment rate: 8.57%


## 7. Monthly Roll-Up

Collapse to one row per `(SK_ID_CURR, MONTHS_BALANCE)` by summing payment
amounts and taking the worst-case (max) of DPD-style metrics.

In [7]:
sum_cols = ['AMT_PAYMENT', 'AMT_INSTALMENT', 'INSTALMENT_VERSION_CHANGE']
max_cols  = ['LATE_PAYMENT_DAYS', 'PAYMENT_AMT_TO_INSTALLMENT_RATIO', 'LATE_PAYMENT_FLAG']

# FIX: flat Index after scalar agg — prefix directly, no ''.join() needed
Payments_Mon_Sum = (
    Payments.groupby(['SK_ID_CURR', 'MONTHS_BALANCE'])[sum_cols]
    .sum()
    .reset_index()
)
Payments_Mon_Sum.columns = [
    'PMNT_INSTLMT_' + col if col not in ('SK_ID_CURR', 'MONTHS_BALANCE') else col
    for col in Payments_Mon_Sum.columns
]

Payments_Mon_Max = (
    Payments.groupby(['SK_ID_CURR', 'MONTHS_BALANCE'])[max_cols]
    .max()
    .reset_index()
)
Payments_Mon_Max.columns = [
    'PMNT_INSTLMT_' + col if col not in ('SK_ID_CURR', 'MONTHS_BALANCE') else col
    for col in Payments_Mon_Max.columns
]

Payments_Mon = Payments_Mon_Sum.merge(
    Payments_Mon_Max, on=['SK_ID_CURR', 'MONTHS_BALANCE'], how='left'
)
del Payments_Mon_Sum, Payments_Mon_Max

print(f"Payments_Mon shape: {Payments_Mon.shape}")

Payments_Mon shape: (8199221, 8)


## 8. Loan-Level Aggregations

Summarise each `(SK_ID_CURR, SK_ID_PREV)` loan into a single row, then roll up
to customer level.

> **Note on `PMNT_INSTLMT_AVG_INSTALLMENT_FREQ`:** `DAYS_ENTRY_PAYMENT` (When was the installments of previous credit actually paid) can be
> out of order relative to `NUM_INSTALMENT_NUMBER` in some loans. The lambda
> self-corrects by calling `.sort_values()` on the unique payment days before
> computing `.diff().mean()`, so negative frequencies are not expected here.
> The feature measures average *days between unique payment dates* for each loan.

In [8]:
Payments_Agg_PrevID = (
    Payments.groupby(['SK_ID_CURR', 'SK_ID_PREV'])
    .agg(
        PMNT_INSTLMT_VRSN_CHANGES_COUNT=('NUM_INSTALMENT_VERSION', 'nunique'),
        PMNT_INSTLMT_DURATION_DAYS=('DAYS_ENTRY_PAYMENT', lambda x: x.max() - x.min() + 1),
        PMNT_INSTLMT_AVG_INSTALLMENT_FREQ=('DAYS_ENTRY_PAYMENT',
            lambda x: (
                pd.Series(x.unique()).sort_values().diff().mean()
                if len(x) > 1 else np.nan
            ),
        ),
        PMNT_INSTLMT_NUM_INSTALMENTS=('NUM_INSTALMENT_NUMBER', 'count'),
    )
    .reset_index()
)

# ── All-time customer-level rollup ────────────────────────────────────────
Payments_Agg_PrevID_counts = (
    Payments_Agg_PrevID
    .groupby('SK_ID_CURR')
    .agg({
        'PMNT_INSTLMT_VRSN_CHANGES_COUNT': ['mean', 'max'],
        'PMNT_INSTLMT_DURATION_DAYS': ['mean', 'max'],
        'PMNT_INSTLMT_AVG_INSTALLMENT_FREQ': ['mean', 'max'],
        'PMNT_INSTLMT_NUM_INSTALMENTS': ['mean', 'max'],
    })
    .reset_index()  # FIX: bring SK_ID_CURR out of index
)
Payments_Agg_PrevID_counts.columns = [
    '_'.join(col).strip().upper() if col[1] else col[0]
    for col in Payments_Agg_PrevID_counts.columns
]

# ── Last-1Y customer-level rollup ─────────────────────────────────────────
Payments_Agg_PrevID_Last1Y = (
    Payments[Payments['MONTHS_BALANCE'] >= -12]
    .groupby(['SK_ID_CURR', 'SK_ID_PREV'])
    .agg(
        PMNT_INSTLMT_VRSN_CHANGES_COUNT_LAST1Y=('NUM_INSTALMENT_VERSION', 'nunique'),
        PMNT_INSTLMT_AVG_INSTALLMENT_FREQ_LAST1Y=('DAYS_ENTRY_PAYMENT', lambda x: ( pd.Series(x.unique()).sort_values().diff().mean() if len(x) > 1 else np.nan ),
        ),
    )
    .reset_index()
)
Payments_Agg_PrevID_Last1Y_counts = (
    Payments_Agg_PrevID_Last1Y
    .groupby('SK_ID_CURR')
    .agg({
        'PMNT_INSTLMT_VRSN_CHANGES_COUNT_LAST1Y':  ['mean', 'max'],
        'PMNT_INSTLMT_AVG_INSTALLMENT_FREQ_LAST1Y': ['mean', 'max'],
    })
    .reset_index()  # FIX: bring SK_ID_CURR out of index
)
Payments_Agg_PrevID_Last1Y_counts.columns = [
    '_'.join(col).strip().upper() if col[1] else col[0]
    for col in Payments_Agg_PrevID_Last1Y_counts.columns
]

# ── Last-6M customer-level rollup ─────────────────────────────────────────
Payments_Agg_PrevID_Last6M = (
    Payments[Payments['MONTHS_BALANCE'] >= -6]
    .groupby(['SK_ID_CURR', 'SK_ID_PREV'])
    .agg(
        PMNT_INSTLMT_VRSN_CHANGES_COUNT_LAST6M=('NUM_INSTALMENT_VERSION', 'nunique'),
        PMNT_INSTLMT_AVG_INSTALLMENT_FREQ_LAST6M=('DAYS_ENTRY_PAYMENT', lambda x: (pd.Series(x.unique()).sort_values().diff().mean() if len(x) > 1 else np.nan ),
        ),
    )
    .reset_index()
)
Payments_Agg_PrevID_Last6M_counts = (
    Payments_Agg_PrevID_Last6M
    .groupby('SK_ID_CURR')
    .agg({
        'PMNT_INSTLMT_VRSN_CHANGES_COUNT_LAST6M':  ['mean', 'max'],
        'PMNT_INSTLMT_AVG_INSTALLMENT_FREQ_LAST6M': ['mean', 'max'],
    })
    .reset_index()  # FIX: bring SK_ID_CURR out of index
)
Payments_Agg_PrevID_Last6M_counts.columns = [
    '_'.join(col).strip().upper() if col[1] else col[0]
    for col in Payments_Agg_PrevID_Last6M_counts.columns
]

print(f"All-time rollup shape : {Payments_Agg_PrevID_counts.shape}")
print(f"Last-1Y rollup shape  : {Payments_Agg_PrevID_Last1Y_counts.shape}")
print(f"Last-6M rollup shape  : {Payments_Agg_PrevID_Last6M_counts.shape}")

All-time rollup shape : (291643, 9)
Last-1Y rollup shape  : (216759, 5)
Last-6M rollup shape  : (182254, 5)


In [9]:
del Payments_Agg_PrevID_Last1Y, Payments_Agg_PrevID_Last6M
gc.collect()

0

## 9. PrevLoans Enrichment

Join previous-application metadata to the loan-level table, then aggregate
to customer level.

In [10]:
PrevLoans = pd.read_parquet(PATH_PREV_LOANS)
PrevLoans = PrevLoans.astype({'SK_ID_PREV': 'string', 'SK_ID_CURR': 'string'})

Payments_PrevLoans = Payments_Agg_PrevID.merge(
    PrevLoans, on=['SK_ID_PREV', 'SK_ID_CURR'], how='left'
)

Payments_Agg_PrevLoans = (
    Payments_PrevLoans.groupby('SK_ID_CURR')
    .agg(
        PMNT_INSTLMT_TOTAL_PREV_LOAN=('SK_ID_PREV', 'nunique'),
        PMNT_INSTLMT_AVG_CREDIT_TO_APPL=('CREDIT_TO_APPL', 'mean'),
        PMNT_INSTLMT_ANNUITY_TO_CREDIT_RATIO=('AMT_ANNUITY_TO_CREDIT_RATIO', 'mean'),
        PMNT_INSTLMT_CNT_HIGH_YIELD_GROUP=('NAME_YIELD_GROUP', lambda x: (x == 'high').sum()),
        PMNT_INSTLMT_CNT_MIDDLE_YIELD_GROUP=('NAME_YIELD_GROUP', lambda x: (x == 'middle').sum()),
        PMNT_INSTLMT_CNT_NFLAG_INSURED_ON_APPROVAL=('NFLAG_INSURED_ON_APPROVAL', 'sum'),
        PMNT_INSTLMT_AVG_DAYS_DECISION=('DAYS_DECISION', 'mean'),
    )
    .reset_index() 
)

del Payments_Agg_PrevID, Payments_PrevLoans, PrevLoans
print(f"PrevLoans enrichment shape: {Payments_Agg_PrevLoans.shape}")

PrevLoans enrichment shape: (291643, 8)


## 10. Time-Window Aggregations

Aggregate the monthly table over two rolling windows.

In [11]:
agg_cols = [
    'PMNT_INSTLMT_AMT_PAYMENT',
    'PMNT_INSTLMT_AMT_INSTALMENT',
    'PMNT_INSTLMT_INSTALMENT_VERSION_CHANGE',
    'PMNT_INSTLMT_LATE_PAYMENT_DAYS',
    'PMNT_INSTLMT_PAYMENT_AMT_TO_INSTALLMENT_RATIO',
    'PMNT_INSTLMT_LATE_PAYMENT_FLAG',
]

# ── Last 3 Year ───────────────────────────────────────────────────────────
Payments_Mon_3Y_Agg = (
    Payments_Mon[Payments_Mon['MONTHS_BALANCE'] >= -36]
    .groupby('SK_ID_CURR')[agg_cols]
    .agg(['mean', 'max', 'min'])
    .reset_index()  # FIX: was missing
)
Payments_Mon_3Y_Agg.columns = [
    '_'.join(col).strip().upper() + '_LAST3Y' if col[1] else col[0]
    for col in Payments_Mon_3Y_Agg.columns
]

# ── Last 2 Year ───────────────────────────────────────────────────────────
Payments_Mon_2Y_Agg = (
    Payments_Mon[Payments_Mon['MONTHS_BALANCE'] >= -24]
    .groupby('SK_ID_CURR')[agg_cols]
    .agg(['mean', 'max', 'min'])
    .reset_index()  # FIX: was missing
)
Payments_Mon_2Y_Agg.columns = [
    '_'.join(col).strip().upper() + '_LAST2Y' if col[1] else col[0]
    for col in Payments_Mon_2Y_Agg.columns
]

# ── Last 1 Year ───────────────────────────────────────────────────────────
Payments_Mon_1Y_Agg = (
    Payments_Mon[Payments_Mon['MONTHS_BALANCE'] >= -12]
    .groupby('SK_ID_CURR')[agg_cols]
    .agg(['mean', 'max', 'min'])
    .reset_index()  # FIX: was missing
)
Payments_Mon_1Y_Agg.columns = [
    '_'.join(col).strip().upper() + '_LAST1Y' if col[1] else col[0]
    for col in Payments_Mon_1Y_Agg.columns
]

# ── Last 6 Months ───────────────────────────────────────────────
Payments_Mon_6M_Agg = (
    Payments_Mon[Payments_Mon['MONTHS_BALANCE'] >= -6]
    .groupby('SK_ID_CURR')[agg_cols]
    .agg(['mean', 'max', 'min'])
    .reset_index()  # FIX: was missing
)
Payments_Mon_6M_Agg.columns = [
    '_'.join(col).strip().upper() + '_LAST6M' if col[1] else col[0]
    for col in Payments_Mon_6M_Agg.columns
]

# ── Last Quarter (3 months) ───────────────────────────────────────────────
Payments_Mon_3M_Agg = (
    Payments_Mon[Payments_Mon['MONTHS_BALANCE'] >= -3]
    .groupby('SK_ID_CURR')[agg_cols]
    .agg(['mean', 'max', 'min'])
    .reset_index()  # FIX: was missing
)
Payments_Mon_3M_Agg.columns = [
    '_'.join(col).strip().upper() + '_LAST3M' if col[1] else col[0]
    for col in Payments_Mon_3M_Agg.columns
]

print(f"3Y agg shape: {Payments_Mon_3Y_Agg.shape}")
print(f"2Y agg shape: {Payments_Mon_2Y_Agg.shape}")
print(f"1Y agg shape: {Payments_Mon_1Y_Agg.shape}")
print(f"6M agg shape: {Payments_Mon_6M_Agg.shape}")
print(f"3M agg shape: {Payments_Mon_3M_Agg.shape}")

3Y agg shape: (264817, 19)
2Y agg shape: (251476, 19)
1Y agg shape: (216759, 19)
6M agg shape: (182254, 19)
3M agg shape: (159944, 19)


## 11. Last-Snapshot Features

The most recent monthly record per customer.

> **Fix 1:** `.last()` is deprecated in pandas ≥ 2.0 (skips NaN rather than
> returning the literal last row). Replaced with `.nth(-1)`.

> **Fix 2:** `MONTHS_BALANCE` is negative; after renaming to
> `PMNT_INSTALMENT_DAYS_SINCE_LAST_TRANSACTION` the value is negated so it
> reads as a positive *'months ago'* count.

In [12]:
# FIX: .nth(-1) returns the actual last row; .last() skips NaN (deprecated)
Payments_Last_Snapshot = (
    Payments_Mon.sort_values('MONTHS_BALANCE')
    .groupby('SK_ID_CURR')[['SK_ID_CURR','MONTHS_BALANCE'] + agg_cols]
    .nth(-1)
    .reset_index()
)
Payments_Last_Snapshot.columns = [
    col + '_LAST_MONTH' if col != 'SK_ID_CURR' else col
    for col in Payments_Last_Snapshot.columns
]
Payments_Last_Snapshot.rename(
    columns={'MONTHS_BALANCE_LAST_MONTH': 'PMNT_INSTALMENT_MONTHS_SINCE_LAST_TRANSACTION'},
    inplace=True,
)
# FIX: negate so the value is a positive 'months ago' count
Payments_Last_Snapshot['PMNT_INSTALMENT_MONTHS_SINCE_LAST_TRANSACTION'] = (
    -Payments_Last_Snapshot['PMNT_INSTALMENT_MONTHS_SINCE_LAST_TRANSACTION']
)

del Payments_Mon
print(f"Last-snapshot shape: {Payments_Last_Snapshot.shape}")

Last-snapshot shape: (291643, 9)


## 12. Merge All Features

The merge spine is `unique_customers` (saved at load time) so no training
customers are silently dropped if they appear in only some feature tables.

Activity-window flags (`PMNT_INSTLMT_IN_LAST_*`) indicate whether a customer
had any instalment activity in that window.

In [13]:
def flag_merge(left, right, on, flag_col):
    """Left-merge and record whether the row matched (1) or not (0)."""
    merged = left.merge(right, on=on, how='left', indicator=True)
    merged[flag_col] = (merged['_merge'] == 'both').astype(int)
    return merged.drop(columns=['_merge'])


# FIX: use unique_customers as spine (not Payments_Agg_PrevID_counts)
Payments_Agg_final = unique_customers.copy()

Payments_Agg_final = Payments_Agg_final.merge(
    Payments_Agg_PrevID_counts, on='SK_ID_CURR', how='left'
)
Payments_Agg_final = Payments_Agg_final.merge(
    Payments_Agg_PrevID_Last1Y_counts, on='SK_ID_CURR', how='left'
)
Payments_Agg_final = Payments_Agg_final.merge(
    Payments_Agg_PrevID_Last6M_counts, on='SK_ID_CURR', how='left'
)
Payments_Agg_final = flag_merge(
    Payments_Agg_final, Payments_Mon_3Y_Agg, 'SK_ID_CURR', 'PMNT_INSTLMT_IS_IN_LAST_3Y'
)
Payments_Agg_final = flag_merge(
    Payments_Agg_final, Payments_Mon_2Y_Agg, 'SK_ID_CURR', 'PMNT_INSTLMT_IS_IN_LAST_2Y'
)
Payments_Agg_final = flag_merge(
    Payments_Agg_final, Payments_Mon_1Y_Agg, 'SK_ID_CURR', 'PMNT_INSTLMT_IS_IN_LAST_1Y'
)
Payments_Agg_final = flag_merge(
    Payments_Agg_final, Payments_Mon_6M_Agg, 'SK_ID_CURR', 'PMNT_INSTLMT_IS_IN_LAST_6M'
)
Payments_Agg_final = flag_merge(
    Payments_Agg_final, Payments_Mon_3M_Agg, 'SK_ID_CURR', 'PMNT_INSTLMT_IS_IN_LAST_3M'
)
Payments_Agg_final = Payments_Agg_final.merge(
    Payments_Last_Snapshot, on='SK_ID_CURR', how='left'
)
Payments_Agg_final = Payments_Agg_final.merge(
    Payments_Agg_PrevLoans, on='SK_ID_CURR', how='left'
)

print(f"Final shape: {Payments_Agg_final.shape}")
Payments_Agg_final.head(3)

Final shape: (291643, 127)


,SK_ID_CURR,PMNT_INSTLMT_VRSN_CHANGES_COUNT_MEAN,PMNT_INSTLMT_VRSN_CHANGES_COUNT_MAX,PMNT_INSTLMT_DURATION_DAYS_MEAN,PMNT_INSTLMT_DURATION_DAYS_MAX,PMNT_INSTLMT_AVG_INSTALLMENT_FREQ_MEAN,PMNT_INSTLMT_AVG_INSTALLMENT_FREQ_MAX,PMNT_INSTLMT_NUM_INSTALMENTS_MEAN,PMNT_INSTLMT_NUM_INSTALMENTS_MAX,PMNT_INSTLMT_VRSN_CHANGES_COUNT_LAST1Y_MEAN,...,PMNT_INSTLMT_LATE_PAYMENT_DAYS_LAST_MONTH,PMNT_INSTLMT_PAYMENT_AMT_TO_INSTALLMENT_RATIO_LAST_MONTH,PMNT_INSTLMT_LATE_PAYMENT_FLAG_LAST_MONTH,PMNT_INSTLMT_TOTAL_PREV_LOAN,PMNT_INSTLMT_AVG_CREDIT_TO_APPL,PMNT_INSTLMT_ANNUITY_TO_CREDIT_RATIO,PMNT_INSTLMT_CNT_HIGH_YIELD_GROUP,PMNT_INSTLMT_CNT_MIDDLE_YIELD_GROUP,PMNT_INSTLMT_CNT_NFLAG_INSURED_ON_APPROVAL,PMNT_INSTLMT_AVG_DAYS_DECISION
0,100002,2.000000,2,539.000000,539.0,29.888889,29.888889,19.000000,19,2.0,...,-24.0,1.0,0,1,1.000000,0.051670,0,0,0.0,-606.0
1,100003,1.333333,2,220.666667,340.0,29.661616,30.818182,8.333333,12,NaN,...,-8.0,1.0,0,3,1.057664,0.126383,0,2,2.0,-1305.0
2,100004,2.000000,2,69.000000,69.0,34.000000,34.000000,3.000000,3,NaN,...,-3.0,1.0,0,1,0.828021,0.266450,0,1,0.0,-815.0


**Recent vs. Historical Ratio:** MEAN_6M / MEAN_3Y (If this ratio is > 1 for DPD for example, it tells the model the customer is becoming increasingly late recently compared to their historical average).

In [18]:
def add_period_ratio_features(df, attributes, numerator_period='6M', denominator_period='3Y'):
    """
    Add a '{attr}_LAST{numerator_period}_TO_{denominator_period}_RATIO' column for
    each attr in *attributes*, computed as
        df[f'{attr}_LAST{numerator_period}'] / df[f'{attr}_LAST{denominator_period}']
    with the denominator guarded against 0 (-> NaN), matching the np.where(denom
    != 0, ..., np.nan) pattern used everywhere else in this notebook.

    Both df[f'{attr}_LAST{numerator_period}'] and df[f'{attr}_LAST{denominator_period}']
    must already exist -- see Section 11's build_window_agg, which is what
    produces the _LAST{period} columns this function reads.
    """
    for attr in attributes:
        numerator_col = f'{attr}_LAST{numerator_period}'
        denominator_col = f'{attr}_LAST{denominator_period}'
        ratio_col = f'{attr}_LAST{numerator_period}_TO_{denominator_period}_RATIO'
        df[ratio_col] = np.where(
            df[denominator_col] != 0,
            df[numerator_col] / df[denominator_col],
            np.nan,
        )
    return df

In [22]:
recent_vs_historical_attrs = [
    'PMNT_INSTLMT_LATE_PAYMENT_FLAG_MEAN',
    'PMNT_INSTLMT_PAYMENT_AMT_TO_INSTALLMENT_RATIO_MEAN',
    'PMNT_INSTLMT_LATE_PAYMENT_DAYS_MEAN',
    'PMNT_INSTLMT_INSTALMENT_VERSION_CHANGE_MEAN',
    'PMNT_INSTLMT_AMT_INSTALMENT_MEAN',
    'PMNT_INSTLMT_AMT_PAYMENT_MEAN'
]

Payments_Agg_final = add_period_ratio_features(
    Payments_Agg_final, recent_vs_historical_attrs,
    numerator_period='6M', denominator_period='3Y',
)

## 13. Quality Checks

In [23]:
null_rates = Payments_Agg_final.isna().mean().sort_values(ascending=False)
print("Null rate (top 15 columns):")
print(null_rates.head(15).to_string())

n_dupes = Payments_Agg_final['SK_ID_CURR'].duplicated().sum()
assert n_dupes == 0, f"Duplicate SK_ID_CURR found: {n_dupes}"
print(f"\nNo duplicate SK_ID_CURR. Final row count: {len(Payments_Agg_final):,}")

Null rate (top 15 columns):
PMNT_INSTLMT_LATE_PAYMENT_FLAG_MEAN_LAST6M_TO_3Y_RATIO            0.725284
PMNT_INSTLMT_INSTALMENT_VERSION_CHANGE_MEAN_LAST6M_TO_3Y_RATIO    0.634515
PMNT_INSTLMT_PAYMENT_AMT_TO_INSTALLMENT_RATIO_MIN_LAST3M          0.451693
PMNT_INSTLMT_PAYMENT_AMT_TO_INSTALLMENT_RATIO_MAX_LAST3M          0.451693
PMNT_INSTLMT_LATE_PAYMENT_DAYS_MIN_LAST3M                         0.451693
PMNT_INSTLMT_LATE_PAYMENT_DAYS_MEAN_LAST3M                        0.451693
PMNT_INSTLMT_PAYMENT_AMT_TO_INSTALLMENT_RATIO_MEAN_LAST3M         0.451693
PMNT_INSTLMT_LATE_PAYMENT_DAYS_MAX_LAST3M                         0.451693
PMNT_INSTLMT_LATE_PAYMENT_FLAG_MAX_LAST3M                         0.451576
PMNT_INSTLMT_AMT_INSTALMENT_MEAN_LAST3M                           0.451576
PMNT_INSTLMT_LATE_PAYMENT_FLAG_MIN_LAST3M                         0.451576
PMNT_INSTLMT_INSTALMENT_VERSION_CHANGE_MIN_LAST3M                 0.451576
PMNT_INSTLMT_LATE_PAYMENT_FLAG_MEAN_LAST3M                        0.4515

## 14. Save Output

In [24]:
Payments_Agg_final.to_parquet(PATH_OUTPUT, index=False)
print(f"Saved → {PATH_OUTPUT}")

Saved → a:\Dominic\My Projects\Home Credit\data\Model-Training\Staging\payment_installments_processed.parquet


## 15. Final Cleanup

In [25]:
del (
    Payments, Payments_Agg_PrevID_counts, Payments_Agg_PrevID_Last1Y_counts,
    Payments_Mon_1Y_Agg, Payments_Mon_3M_Agg, Payments_Last_Snapshot,
    Payments_Agg_PrevLoans, unique_customers, base,
)
gc.collect()
print("All DataFrames released.")

All DataFrames released.
